In [ ]:
import numpy as np
import xtrack as xt
import xobjects as xo
import xpart as xp
from tqdm import tqdm
from scipy import constants 
import xfields as xf

line = xt.Line.from_json('sps.json')
particle_ref=line.particle_ref

# context = xo.ContextCupy()
context = xo.ContextCpu(omp_num_threads=4)
line.build_tracker(_context=context)
twiss=line.twiss()

clight=constants.speed_of_light
circumference = line.get_length()

#gamma0=190.99019102

from ion_properties import lead,calcium,xenon

for ion in [lead,calcium]:

    # Ion properties:
    q0 = ion.q0
    mass0 = ion.mass0

    gamma = ion.gamma_cooling
    beta= ion.beta_rel
    p0c = mass0*gamma*beta #eV/c

    bunch_intensity = ion.bunch_intensity

    particle_ref = xp.Particles(p0c=p0c, mass0=mass0, q0=q0, gamma0=gamma)

    line.particle_ref=particle_ref

    nemitt = 1.5e-6 # m*rad (normalized emittance)
    sigma_z = 0.063 # m
    sigma_z = ion.bunch_length


    emittance=nemitt/(beta*gamma)

    num_particles=int(1e4)

    particles = xp.generate_matched_gaussian_bunch(
            num_particles=num_particles,
            total_intensity_particles=bunch_intensity,
            nemitt_x=nemitt, nemitt_y=nemitt, sigma_z=sigma_z,
            particle_ref=particle_ref,
            line=line,        
            )

    particles._init_random_number_generator()    
    particles0=particles.copy()
    # sigma_dp=2e-4  
    sigma_dp=np.std(particles.delta)

    ##################
    # Laser Cooler #
    ##################

    #laser-ion beam collision angle
    theta_l = 2.6*np.pi/180 # rad
    #theta_l = ion.theta_l
    nx = 0; ny = -np.sin(theta_l); nz = -np.cos(theta_l)

    # Ion excitation energy:
    ion_excited_lifetime=ion.excited_lifetime
    hw0 = ion.hw0 # eV
    hc=constants.hbar*clight/constants.e # eV*m (hbar c)
    lambda_0 = 2*np.pi*hc/hw0 # m -- ion excitation wavelength

    lambda_l = lambda_0*gamma*(1 + beta*np.cos(theta_l)) # m -- laser wavelength

    # Shift laser wavelength for fast longitudinal cooling:
    #lambda_l = lambda_l*(1+1*sigma_dp) # m

    lambda_l = ion.lambda_l

    laser_frequency = clight/lambda_l # Hz
    sigma_w = 2*np.pi*laser_frequency*sigma_dp
    #sigma_w = 2*np.pi*laser_frequency*sigma_dp/2 # for fast longitudinal cooling

    sigma_t = 1/sigma_w # sec -- Fourier-limited laser pulse
    print('Laser pulse duration sigma_t = %.2f ps' % (sigma_t/1e-12))
    print('Laser wavelength = %.2f nm' % (lambda_l/1e-9))

    laser_waist_radius = 1.3e-3 #m
    laser_energy = 5e-3
    #laser_energy = ion.pulse_energy
    laser_x = ion.laser_x

    GF_IP = xt.PulsedLaser(
                    laser_x=laser_x,
                    laser_y=0,
                    laser_z=0,
                    
                    laser_direction_nx = 0,
                    laser_direction_ny = ny,
                    laser_direction_nz = nz,
                    laser_energy         = laser_energy, # J
                    laser_duration_sigma = sigma_t, # sec
                    laser_wavelength = lambda_l, # m
                    laser_waist_radius = laser_waist_radius, # m
                    laser_waist_shift = 0, # m
                    ion_excitation_energy = hw0, # eV
                    ion_excited_lifetime  = ion_excited_lifetime, # sec                   
                    )

    # simulation parameters: simulate 10 s of cooling, and take data once every 100 ms
    max_time_s = 0.05
    int_time_s = 0.001
    T_per_turn = circumference/(clight*beta)
    num_turns = int(max_time_s/T_per_turn)
    save_interval = int(int_time_s/T_per_turn)

    # create a monitor object, to reduce holded data
    monitor = xt.ParticlesMonitor(start_at_turn=0, stop_at_turn=1,
                                n_repetitions=int(num_turns/save_interval),
                                repetition_period=save_interval,
                                num_particles=num_particles)

    #ibs_kick = xf.IBSKineticKick(num_slices=50)
    #line.configure_intrabeam_scattering(
    #element=ibs_kick, name="ibskick", index=-1, update_every=50)


    line.discard_tracker()
    IP_index=16675   
    line.insert_element('monitor', element=monitor, index=IP_index)
    line.insert_element('GF_IP', element=GF_IP, index=IP_index) #this way monitor comes after the laser

    # at interaction points: #from https://anaconda.org/petrenko/li_like_ca_in_sps/notebook
    beta_x  =  twiss.betx[IP_index]
    beta_y  =  twiss.bety[IP_index]
    alpha_x =  twiss.alfx[IP_index]
    alpha_y =  twiss.alfy[IP_index]

    gamma_x=twiss.gamx[IP_index]
    gamma_y=twiss.gamy[IP_index]

    Dx  =  twiss.dx[IP_index]
    Dpx =  twiss.dpx[IP_index]

    Dy  =  twiss.dy[IP_index]
    Dpy =  twiss.dpy[IP_index]

    particles=particles0.copy()

    #context = xo.ContextCpu(omp_num_threads='auto')
    line.build_tracker(_context=context)
    #line.optimize_for_tracking()
   

    line.track(particles, num_turns=num_turns,
                turn_by_turn_monitor=False,with_progress=True)

    # extract relevant values
    x = monitor.x[:,:,0]
    px = monitor.px[:,:,0]
    y = monitor.y[:,:,0]
    py = monitor.py[:,:,0]
    delta = monitor.delta[:,:,0]
    zeta = monitor.zeta[:,:,0]
    state = monitor.state[:,:,0]
    time = monitor.at_turn[:, 0, 0] * T_per_turn

    action_x = (gamma_x*(x-Dx*delta)**2 + 2*alpha_x*(x-Dx*delta)*(px-Dpx*delta)+ beta_x*(px-Dpx*delta)**2)
    action_y = (gamma_y*(y-Dy*delta)**2 + 2*alpha_y*(y-Dy*delta)*(py-Dpy*delta)+ beta_y*(py-Dpy*delta)**2)

    emittance_x_twiss=np.mean(action_x,axis=1)*gamma/2

    np.savez(f'results/cooling_rates/transverse/{ion.name}.npz', x=x, px=px, y=y, py=py, zeta=zeta, delta=delta,
            action_x=action_x,action_y=action_y,emittance_x=emittance_x_twiss,
            state=state, time=time,s_per_turn=T_per_turn)

Loading line from dict:   0%|          | 0/38786 [00:00<?, ?it/s]

Done loading line from dict.           
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
*** Maximum RMS bunch length 0.23410655471239558m.
... distance to target bunch length: -6.3000e-02
... distance to target bunch length: 1.6464e-01
... distance to target bunch length: 7.8245e-02
... distance to target bunch length: -3.7042e-03
... distance to target bunch length: -7.4446e-05
... distance to target bunch length: -1.7696e-08
... distance to target bunch length: 3.0957e-07
--> Bunch length: 0.06299998230379812
--> Emittance: 0.12257873919401163
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Laser pulse duration sigma_t = 2.79 ps
Laser wavelength = 1031.80 nm
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Tracking:   0%|          | 0/2200 [00:00<?, ?it/s]

*** Maximum RMS bunch length 0.23411645187427735m.


/home/pkruyt/miniforge3/envs/xsuite-laser2024/lib/python3.12/site-packages/scipy/integrate/_quadpack_py.py:1272: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,


... distance to target bunch length: -9.3234e-02
... distance to target bunch length: 1.3265e-01
... distance to target bunch length: 9.6426e-02
... distance to target bunch length: 5.5382e-03
... distance to target bunch length: -6.2242e-04
... distance to target bunch length: -5.3543e-06
... distance to target bunch length: 8.6936e-11
... distance to target bunch length: -5.2310e-07
--> Bunch length: 0.0950000000869361
--> Emittance: 0.3669203616089402
Laser pulse duration sigma_t = 2.00 ps
Laser wavelength = 768.00 nm
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Tracking:   0%|          | 0/2200 [00:00<?, ?it/s]